In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import openai
import json
from threading import Lock

open_ai_keys = [
    os.getenv("OPENAI_KEY_1"),
    os.getenv("OPENAI_KEY_2"),
    os.getenv("OPENAI_KEY_3"),
    os.getenv("OPENAI_KEY_4"),
    os.getenv("OPENAI_KEY_5"),
    os.getenv("OPENAI_KEY_6"),
    os.getenv("OPENAI_KEY_7"),
    os.getenv("OPENAI_KEY_8"),
    os.getenv("OPENAI_KEY_9"),
    os.getenv("OPENAI_KEY_10"),
    os.getenv("OPENAI_KEY_11"),
    os.getenv("OPENAI_KEY_21"),
    os.getenv("OPENAI_KEY_13"),
    os.getenv("OPENAI_KEY_14"),
    os.getenv("OPENAI_KEY_15"),
    os.getenv("OPENAI_KEY_16"),
    os.getenv("OPENAI_KEY_17"),
    os.getenv("OPENAI_KEY_18"),
    os.getenv("OPENAI_KEY_19"),
    os.getenv("OPENAI_KEY_20"),
    os.getenv("OPENAI_KEY_21"),
    os.getenv("OPENAI_KEY_22"),
    os.getenv("OPENAI_KEY_23"),
    os.getenv("OPENAI_KEY_24"),
    os.getenv("OPENAI_KEY_25")
]

In [2]:

executor = ThreadPoolExecutor(max_workers=25)

In [25]:
def epl_prompt_context(previous=None, current=None, next=None):
    if not current: # Ensure that there is a current sentence
        return "Error: A current sentence is required."

    # Initialize content with the current sentence
    content = f"Analyze this phrase for Ethos (E), Pathos (P), and Logos (L) from 0.0 - 9.9: '{current}'. Return a compact dictionary format like this: 'E: x.x, P: y.y, L: z.z, E_Explanation: "", P_Explanation: "", L_Explanation: "". Explanations must be 1 sentence long."
    
    if previous:
        content = f"Given the previous sentence: '{previous}', " + content
    if next:
        content = content + f" Also, consider the following sentence: '{next}'."

    prompt = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[
            {
                "role": "system",
                "content": "You are a master debater and assistant to the judge of the US Presidential Debates."
            },
            {
                "role": "user",
                "content": content
            }
        ]
    )
    return prompt

In [26]:
def process_lines(lines):
    """
    Processes the transcript lines into speaker turns.
    Each speaker turn starts with a line that has a colon ':'.
    """
    speaker_turns = []
    current_turn = []

    for line in lines:
        if ':' in line:  # This line is the start of a new speaker turn
            speaker_turns.append(' '.join(current_turn))  # Add the current turn to the turns list
            current_turn = [line]  # Start a new turn with this line
        else:
            current_turn.append(line)  # This line is part of the current turn

    speaker_turns.append(' '.join(current_turn))  # Add the last turn

    # The first element is empty, so we skip it
    speaker_turns = speaker_turns[1:]

    # Write speaker turns to file
    with open("speaker_turns.txt", "w") as file:
        for i, turn in enumerate(speaker_turns):
            file.write(f"Speaker turn {i+1}:\n{turn}\n\n")

    return speaker_turns

In [5]:
def get_turns(speaker_turns, i):
    """
    Returns the previous, current, and next turns given a list of speaker turns and an index.
    """
    prev_turn = speaker_turns[i - 1] if i > 0 else ''  # If this is the first turn, there is no previous turn
    curr_turn = speaker_turns[i]
    next_turn = speaker_turns[i + 1] if i < len(speaker_turns) - 1 else ''  # If this is the last turn, there is no next turn

    return prev_turn, curr_turn, next_turn

In [6]:
def threaded_process(i, speaker_turns, output_file_full, output_file_content, lock1, lock2, key):
    # Get previous, current and next turn
    prev_turn, curr_turn, next_turn = get_turns(speaker_turns, i)
    
    # Set the API key
    openai.api_key = key
    
    # Call OpenAI API and get response
    response = epl_prompt_context(prev_turn, curr_turn, next_turn)
    
    # Write complete response to JSON file
    lock1.acquire()
    try:
        with open(output_file_full, 'a') as outfile:
            json.dump(response, outfile)
            outfile.write('\n')  # Write newline character after each response
    finally:
        lock1.release()

    # Extract 'content' field and write to second JSON file
    content = response['choices'][0]['message']['content']
    
    # Include the turns that were passed to GPT in the content
    content_with_turns = {
        "prev_turn": prev_turn,
        "curr_turn": curr_turn,
        "next_turn": next_turn,
        "content": content
    }
    
    lock2.acquire()
    try:
        with open(output_file_content, 'a') as outfile:
            json.dump(content_with_turns, outfile)  # Write the content with turns instead of just content
            outfile.write('\n')  # Write newline character after each response
    finally:
        lock2.release()

In [7]:
def read_transcript_and_analyze(input_file, output_file_full, output_file_content, startline=None, endline=None, startturn=None, endturn=None, runcount=None):
    # Error checking for parameters
    if startline is not None and startturn is not None:
        raise ValueError("Parameters 'startline' and 'startturn' are mutually exclusive. Use only 'startline' to specify the line number to start analysis, or 'startturn' to specify the speaker turn number to start analysis.")
    if endline is not None and endturn is not None:
        raise ValueError("Parameters 'endline' and 'endturn' are mutually exclusive. Use only 'endline' to specify the line number to end analysis, or 'endturn' to specify the speaker turn number to end analysis.")
    if startline is not None or endline is not None or startturn is not None or endturn is not None:
        if runcount is not None:
            raise ValueError("Parameter 'runcount' cannot be used together with line or turn parameters. Use 'runcount' alone or with either 'startline' and 'endline', or 'startturn' and 'endturn'.")
    
    with open(input_file, 'r') as file:
        lines = file.readlines()

    # Process lines and extract speaker turns
    speaker_turns = process_lines(lines)
    
    # Determine start and end points for analysis
    if startline is not None:
        start = startline
    elif startturn is not None:
        start = startturn
    else:
        start = 0

    if endline is not None:
        end = endline
    elif endturn is not None:
        end = endturn
    else:
        end = len(speaker_turns)  # Updated this line to consider turns instead of lines

    # Determine number of runs
    if runcount is not None:
        end = min(start + runcount, len(speaker_turns))  # Updated this line to consider turns instead of lines

    # Check if 'end' exceeds the length of the speaker turns
    if end > len(speaker_turns):  # Updated this line to consider turns instead of lines
        end = len(speaker_turns)  # Updated this line to consider turns instead of lines
        print(f"Warning: The end point exceeded the length of the speaker turns. It has been adjusted to the last speaker turn: {end}")

    # Create locks for writing to files
    lock1 = Lock()
    lock2 = Lock()

    # Use a ThreadPoolExecutor to process the lines concurrently
    with ThreadPoolExecutor(max_workers=25) as executor:
        # Assign API keys cyclically
        futures = [executor.submit(threaded_process, i, speaker_turns, output_file_full, output_file_content, lock1, lock2, open_ai_keys[i % len(open_ai_keys)]) for i in range(start, end)]
        
        # Ensure all threads have finished
        for future in as_completed(futures):
            pass


In [8]:
        
read_transcript_and_analyze(input_file="1992-10-19-debate-preclean.txt", 
                    output_file_full="92-test-epl-full-run.json", 
                    output_file_content="92-test-epl-full-run-content.json")

In [10]:
import pandas as pd
import json
import re

def process_jsonl_to_df(filename):
    with open(filename, 'r') as file:
        data = [json.loads(line) for line in file]

    df = pd.json_normalize(data)

    # Extract the scores and explanations from the 'choices' column
    df['content'] = df['choices'].apply(lambda x: x[0]['message']['content'])

    def extract_score(content, category):
        match = re.search(f'{category}: ([\d.]+)', content)
        return float(match.group(1)) if match else None

    def extract_explanation(content, category):
        match = re.search(f'{category}_Explanation: (.+?)(?=[EPL]_Explanation|$)', content)
        return match.group(1).strip() if match else None

    df['Ethos_Score'] = df['content'].apply(lambda x: extract_score(x, 'Ethos'))
    df['Pathos_Score'] = df['content'].apply(lambda x: extract_score(x, 'Pathos'))
    df['Logos_Score'] = df['content'].apply(lambda x: extract_score(x, 'Logos'))

    df['E_Explanation'] = df['content'].apply(lambda x: extract_explanation(x, 'E'))
    df['P_Explanation'] = df['content'].apply(lambda x: extract_explanation(x, 'P'))
    df['L_Explanation'] = df['content'].apply(lambda x: extract_explanation(x, 'L'))

    # Drop the original 'content' column
    df = df.drop(columns=['content'])

    return df

In [12]:
costs = process_jsonl_to_df("../1992-run/92-test-epl-full-run.json")
costs

,id,object,created,model,choices,usage.prompt_tokens,usage.completion_tokens,usage.total_tokens,Ethos_Score,Pathos_Score,Logos_Score,E_Explanation,P_Explanation,L_Explanation
0,chatcmpl-7Pfz5B5mrF0NqIuoeD7FKUXB7PfjC,chat.completion,1686353943,gpt-4-0314,"[{'message': {'role': 'assistant', 'content': ...",259,110,369,7.5,6.2,7.0,Perot establishes his credibility by stating h...,Perot appeals to the audience's emotions by de...,Perot uses the information about the populatio...
1,chatcmpl-7Pfz5BP2QWTuQO4odiZB6ZM8YkbhB,chat.completion,1686353943,gpt-4-0314,"[{'message': {'role': 'assistant', 'content': ...",192,84,276,6.8,2.0,4.0,President Bush establishes a moderate level of...,He slightly evokes emotions by expressing unce...,There is a weak logical element as he demonstr...
2,chatcmpl-7Pfz5Z67gf8gEsdu3S9xLkcpJIqg5,chat.completion,1686353943,gpt-4-0314,"[{'message': {'role': 'assistant', 'content': ...",484,94,578,7.1,8.4,7.9,Bush references his own experience during the ...,He appeals to the emotions and fears of the au...,Bush uses logical argument and statistics abou...
3,chatcmpl-7Pfz50fSyfYsQbl2UZirRWybTEvBk,chat.completion,1686353943,gpt-4-0314,"[{'message': {'role': 'assistant', 'content': ...",187,70,257,7.0,3.5,8.0,Clinton uses his authoritative position to sho...,The focus on job growth appeals to the audienc...,"He cites a specific, relevant statistic to mak..."
4,chatcmpl-7Pfz5Wvd7Wbo9aZKstFD7xTi4mE1i,chat.completion,1686353943,gpt-4-0314,"[{'message': {'role': 'assistant', 'content': ...",241,86,327,3.2,6.8,4.3,Lehrer's authority as a moderator contributes ...,Lehrer's question appeals to the audience's cu...,The question employs Logos by seeking a reason...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207,chatcmpl-7Pg0z6WnByDP719eknsTXlI8pGKHq,chat.completion,1686354061,gpt-4-0314,"[{'message': {'role': 'assistant', 'content': ...",639,99,738,8.1,8.7,7.8,PEROT establishes credibility by referencing h...,"He evokes emotions of patriotism, empowerment,...",He uses logical arguments about qualifications...
208,chatcmpl-7Pg0mxdfAgTggFPHQEHzG4uQQ3PK5,chat.completion,1686354048,gpt-4-0314,"[{'message': {'role': 'assistant', 'content': ...",551,267,818,0.0,0.0,0.0,The phrase doesn't provide any credibility or ...,"There is no emotional appeal in this phrase,",Clinton provides logical reasoning through sta...
209,chatcmpl-7Pg0u5OefhCThKqIti4bL0yo6TpCG,chat.completion,1686354056,gpt-4-0314,"[{'message': {'role': 'assistant', 'content': ...",907,241,1148,5.8,6.3,3.9,The sentence mainly announces the upcoming clo...,The sentence sets up the anticipation for the ...,Clinton provides an argument against trickle-d...
210,chatcmpl-7Pg0YIoHPUE5sAGJAbBnUIdykrfe1,chat.completion,1686354034,gpt-4-0314,"[{'message': {'role': 'assistant', 'content': ...",1013,470,1483,3.5,1.0,0.0,The speaker has some authority as the moderato...,There is minimal emotional appeal in informing...,Logical reasoning is provided to argue against...


In [13]:
import pandas as pd

def calculate_total_cost(df):
    # Assuming df is the DataFrame obtained from the function process_jsonl_to_df

    # Calculate the total tokens
    total_prompt_tokens = df["usage.prompt_tokens"].sum()
    total_response_tokens = df["usage.completion_tokens"].sum()

    # Cost for 1k tokens
    prompt_token_price = 0.03  # This is the price per 1k prompt tokens
    response_token_price = 0.06  # This is the price per 1k response tokens

    # Calculate the total costs
    total_prompt_cost = (total_prompt_tokens / 1000) * prompt_token_price
    total_response_cost = (total_response_tokens / 1000) * response_token_price

    # Total cost
    total_cost = total_prompt_cost + total_response_cost

    # Create a summary dataframe
    summary = pd.DataFrame({
        "total_prompt_tokens": [total_prompt_tokens],
        "prompt_cost": [total_prompt_cost],
        "total_response_tokens": [total_response_tokens],
        "response_cost": [total_response_cost],
        "total_cost": [total_cost]
    })

    return summary

In [15]:
calculate_total_cost(costs)

,total_prompt_tokens,prompt_cost,total_response_tokens,response_cost,total_cost
0,94963,2.84889,24355,1.4613,4.31019


In [ ]:
4.56 after 1 run

In [17]:
import pandas as pd
import re
import json
import numpy as np

def extract_scores_and_explanations(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    scores = []
    explanations = []

    # Regex patterns
    score_pattern = re.compile(r'Ethos: ([0-9.]+), Pathos: ([0-9.]+), Logos: ([0-9.]+)')
    explanation_pattern = re.compile(r'E_Explanation: (.*), P_Explanation: (.*), L_Explanation: (.*)')

    for line in lines:
        content = json.loads(line)['choices'][0]['message']['content']

        # Use regex to locate and extract scores and explanations
        score_match = score_pattern.findall(content)
        explanation_match = explanation_pattern.findall(content)

        # If there are multiple evaluations in a single "content" field, handle them separately
        for score, explanation in zip(score_match, explanation_match):
            scores.append(score)
            explanations.append(explanation)

    # Convert lists of tuples to dataframes
    scores_df = pd.DataFrame(scores, columns=['Ethos_Score', 'Pathos_Score', 'Logos_Score'])
    explanations_df = pd.DataFrame(explanations, columns=['E_Explanation', 'P_Explanation', 'L_Explanation'])

    # Convert score columns to numeric types
    for col in ['Ethos_Score', 'Pathos_Score', 'Logos_Score']:
        scores_df[col] = pd.to_numeric(scores_df[col])

    # Concatenate dataframes
    result_df = pd.concat([scores_df, explanations_df], axis=1)

    return result_df

# Usage
file_path = "92-test-epl-full-run.json"
result_df = extract_scores_and_explanations(file_path)


In [18]:
result_df

,Ethos_Score,Pathos_Score,Logos_Score,E_Explanation,P_Explanation,L_Explanation
0,7.5,6.2,7.0,Perot establishes his credibility by stating h...,Perot appeals to the audience's emotions by de...,Perot uses the information about the populatio...
1,6.8,2.0,4.0,President Bush establishes a moderate level of...,He slightly evokes emotions by expressing unce...,There is a weak logical element as he demonstr...
2,7.1,8.4,7.9,Bush references his own experience during the ...,He appeals to the emotions and fears of the au...,Bush uses logical argument and statistics abou...
3,7.0,3.5,8.0,Clinton uses his authoritative position to sho...,The focus on job growth appeals to the audienc...,"He cites a specific, relevant statistic to mak..."
4,3.2,6.8,4.3,Lehrer's authority as a moderator contributes ...,Lehrer's question appeals to the audience's cu...,The question employs Logos by seeking a reason...
...,...,...,...,...,...,...
222,8.5,9.4,7.9,"Clinton thanks the commission and opponents, s...",Clinton emotionally connects with the audience...,Clinton provides an argument against trickle-d...
223,3.5,1.0,0.0,The speaker has some authority as the moderato...,There is minimal emotional appeal in informing...,There is no logical argument being presented i...
224,6.5,7.0,8.5,Clinton appeals to his credibility as a knowle...,Emotionally engaging language is used to empha...,Logical reasoning is provided to argue against...
225,0.0,0.0,0.0,This phrase only introduces the next speaker,There is no emotional appeal as it only serves...,There is no logical argument presented within ...


In [22]:
import pandas as pd
import json
import re

def parse_json_to_df_2(filename):
    df_list = []
    with open(filename, 'r') as file:
        for line in file:
            item = json.loads(line)
            current_sentence = item.get("curr_turn", "")
            content = item.get("content", "")
            
            # Extract the scores using regex
            ethos_score = re.search('Ethos: (\d.\d)', content)
            pathos_score = re.search('Pathos: (\d.\d)', content)
            logos_score = re.search('Logos: (\d.\d)', content)
            
            # Extract the explanations using regex
            ethos_explanation = re.search('E_Explanation: (.*), P_Explanation', content)
            pathos_explanation = re.search('P_Explanation: (.*), L_Explanation', content)
            logos_explanation = re.search('L_Explanation: (.*)', content)

            # Create a dictionary for the current item
            df_dict = {
                "current_sentence": current_sentence,
                "Ethos": ethos_score.group(1) if ethos_score else None,
                "Pathos": pathos_score.group(1) if pathos_score else None,
                "Logos": logos_score.group(1) if logos_score else None,
                "E_Explanation": ethos_explanation.group(1) if ethos_explanation else None,
                "P_Explanation": pathos_explanation.group(1) if pathos_explanation else None,
                "L_Explanation": logos_explanation.group(1) if logos_explanation else None
            }
            
            df_list.append(df_dict)

    df = pd.DataFrame(df_list)
    return df


In [23]:
scores = parse_json_to_df_2('92-test-epl-full-run-content.json')
scores


,current_sentence,Ethos,Pathos,Logos,E_Explanation,P_Explanation,L_Explanation
0,PEROT: I grew up 5 blocks from Arkansas. Let's...,7.5,6.2,7.0,Perot establishes his credibility by stating h...,Perot appeals to the audience's emotions by de...,Perot uses the information about the populatio...
1,PRESIDENT BUSH: Do I have 1 minute? Just the g...,6.8,2.0,4.0,President Bush establishes a moderate level of...,He slightly evokes emotions by expressing unce...,There is a weak logical element as he demonstr...
2,"BUSH: Well, he doesn't like trickle down gover...",7.1,8.4,7.9,Bush references his own experience during the ...,He appeals to the emotions and fears of the au...,Bush uses logical argument and statistics abou...
3,CLINTON: Mr. Bush's Bureau of Labor Statistics...,7.0,3.5,8.0,Clinton uses his authoritative position to sho...,The focus on job growth appeals to the audienc...,"He cites a specific, relevant statistic to mak..."
4,LEHRER: Why is that a mistake?\n,3.2,6.8,4.3,Lehrer's authority as a moderator contributes ...,Lehrer's question appeals to the audience's cu...,The question employs Logos by seeking a reason...
...,...,...,...,...,...,...,...
207,PEROT: To the millions of fine decent people w...,8.1,8.7,7.8,None,None,He uses logical arguments about qualifications...
208,LEHRER: Mr. Perot.\n,0.0,0.0,0.0,The phrase doesn't provide any credibility or ...,There is no emotional appeal in this phrase,The statement does not offer any logical reaso...
209,"LEHRER: All right. Now, that was the final que...",5.8,6.3,3.9,The sentence mainly announces the upcoming clo...,The sentence sets up the anticipation for the ...,The sentence functions as a logical transition...
210,"LEHRER: Governor Clinton, you have 1 minute.\n",3.5,1.0,0.0,The speaker has some authority as the moderato...,There is minimal emotional appeal in informing...,There is no logical argument being presented i...
